### 📦 Step 1: Install Required Libraries  
This installs deep learning, visualization, and utility libraries like:  
- `tensorflow` 🧠  
- `opencv` 📸  
- `gradio` 🌐  
- `scikit-learn`, `matplotlib`, etc. 📊  
setup).

In [ ]:
!pip install tensorflow numpy matplotlib opencv-python scikit-learn gradio

In [ ]:
import numpy as np  # Importing NumPy for numerical operations and array manipulations
import matplotlib.pyplot as plt  # Importing Matplotlib for plotting graphs and visualizations
import seaborn as sns  # Importing Seaborn for statistical data visualization, built on top of Matplotlib
import tensorflow as tf  # Importing TensorFlow for building and training machine learning models
from tensorflow import keras  # Importing Keras, a high-level API for TensorFlow, to simplify model building
from tensorflow.keras import Layer  # Importing Layer class for creating custom layers in Keras
from tensorflow.keras.models import Sequential  # Importing Sequential model for building neural networks layer-by-layer
from tensorflow.keras.layers import Rescaling , GlobalAveragePooling2D
from tensorflow.keras import layers, optimizers, callbacks  # Importing various modules for layers, optimizers, and callbacks in Keras
from sklearn.utils.class_weight import compute_class_weight  # Importing function to compute class weights for imbalanced datasets
from tensorflow.keras.applications import EfficientNetV2B2  # Importing EfficientNetV2S model for transfer learning
from sklearn.metrics import confusion_matrix, classification_report  # Importing functions to evaluate model performance
import gradio as gr  # Importing Gradio for creating interactive web interfaces for machine learning models

In [ ]:
📚 Step 2: Import All Necessary Libraries
This cell imports the libraries needed for:

Numpy: Numerical operations ➕
Matplotlib & Seaborn: Data & image visualization 📉📷
TensorFlow & Keras: Deep learning framework 🧠
EfficientNet: Pre-trained model for image classification 🔍
Scikit-learn: Evaluation metrics 📊
Gradio: Creating a user interface for model deployment 🌐

In [ ]:
import tensorflow as tf

# ✅ Dataset path
dataset_dir = r"C:\Users\S&P\Downloads\edunet internhsip\garbage_Image_Dataset"

# ✅ Image configuration
image_size = (224, 224)
batch_size = 32
seed = 42

# ✅ Define correct class order
class_names = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

# ✅ Load training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    shuffle=True,
    image_size=image_size,
    batch_size=batch_size,
    class_names=class_names  # 👈 FORCES correct order
)

# ✅ Load validation dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    shuffle=True,
    image_size=image_size,
    batch_size=batch_size,
    class_names=class_names  # 👈 KEEP consistent
)

# ✅ Split validation into validation and test
val_batches = tf.data.experimental.cardinality(val_ds)
test_ds = val_ds.take(val_batches // 2)
val_dat = val_ds.skip(val_batches // 2)

# ✅ Prefetch test dataset
test_ds_eval = test_ds.cache().prefetch(tf.data.AUTOTUNE)

# ✅ Output summary
print("\n📊 Dataset Summary")
print("-" * 40)

print("📁 Class Names:")
for i, cls in enumerate(train_ds.class_names, 1):
    print(f"  {i}. {cls}")

print(f"\n🔢 Total Classes     : {len(train_ds.class_names)}")
print(f"📦 Training Batches  : {len(train_ds)}")
print(f"📦 Validation Batches: {len(val_dat)}")
print(f"📦 Test Batches      : {len(test_ds)}")
print("-" * 40)


In [ ]:
📁 Step 3: Load the Dataset
This section:

Loads the image dataset from your system.
Splits it into ➗:
80% training
10% validation
10% testing
Uses image_dataset_from_directory() for loading images in batches.
Applies caching and prefetching to make model training faster 🚀
Prints useful info like class names and number of batches 🧾


In [ ]:
🖼️ Step 4: Visualize Sample Images
This cell:

Displays up to 12 sample images from the training dataset.
Shows the class label of each image. Useful for checking whether the data is loaded and labeled correctly 👁️✅

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    num_images = min(len(images), 12)
    for i in range(num_images):
        ax = plt.subplot(4, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f"{train_ds.class_names[labels[i]]} ({i})", fontsize=10)
        plt.axis("off")

plt.suptitle("Sample Images from Training Dataset", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])  # adjust to make space for title
plt.show()


In [ ]:
📊 Step 5: Visualize Class Distributions
This cell:

Counts the number of images in each class (Train, Validation, Test) 📦
Calculates percentage distribution of each class
Displays a clean table and creates bar plots 📈 using Seaborn Helps us understand class imbalance (if any) before training the model ⚖️

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Function to count class distribution (in %)
def count_distribution(dataset, class_names):
    total = 0
    counts = {name: 0 for name in class_names}
    
    for _, labels in dataset:
        for label in labels.numpy():
            counts[class_names[label]] += 1
            total += 1

    return {k: round((v / total) * 100, 2) for k, v in counts.items()}

# Improved bar plot function with seaborn
def simple_bar_plot(dist, title):
    plt.figure(figsize=(9, 6))
    sns.set_style("whitegrid")
    
    bars = plt.bar(dist.keys(), dist.values(), color=sns.color_palette("pastel"), edgecolor='black')

    # Add value labels on top
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, height + 1, f'{height}%',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('Percentage (%)', fontsize=12)
    plt.xticks(rotation=45, fontsize=10)
    plt.ylim(0, 100)
    plt.tight_layout()
    plt.show()

# Get class names
class_names = train_ds.class_names

# Compute distributions
train_dist = count_distribution(train_ds, class_names)
val_dist = count_distribution(val_ds, class_names)
test_dist = count_distribution(test_ds, class_names)
overall_dist = {k: round((train_dist[k] + val_dist[k]) / 2, 2) for k in class_names}

# Create DataFrame
dist_df = pd.DataFrame({
    'Class': class_names,
    'Train (%)': [train_dist[k] for k in class_names],
    'Validation (%)': [val_dist[k] for k in class_names],
    'Test (%)': [test_dist[k] for k in class_names],
    'Overall (%)': [overall_dist[k] for k in class_names],
})

# Print clean table
print("\nClass Distribution Summary:\n")
print(dist_df.to_string(index=False))

# Plot distributions
simple_bar_plot(train_dist, "Training Set Class Distribution (%)")
simple_bar_plot(val_dist, "Validation Set Class Distribution (%)")
simple_bar_plot(test_dist, "Test Set Class Distribution (%)")
simple_bar_plot(overall_dist, "Overall Class Distribution (%)")


In [ ]:
### 🔄 Step 6: Data Augmentation  
Applies random image flips, rotations, etc.  
Helps improve generalization and reduce overfitting.  


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Count class occurrences and collect all labels
class_counts = {i: 0 for i in range(len(class_names))}
all_labels = []

for images, labels in train_ds:
    for label in labels.numpy():
        class_counts[label] += 1
        all_labels.append(label)

# Compute class weights (balanced)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(class_names)),
    y=all_labels
)

# Map class index to weight
class_weights = {i: round(w, 4) for i, w in enumerate(class_weights_array)}

# ✨ Display results nicely
print("\n📊 Class Distribution in Training Set")
print("-" * 40)
for i, name in enumerate(class_names):
    print(f"{i}. {name:<10} ➤ Samples: {class_counts[i]:<4} | Weight: {class_weights[i]}")
print("-" * 40)
print(f"\n🧮 Total Training Samples: {sum(class_counts.values())}")


### 🧱 Step 7: Define Model  
Using EfficientNetV2B2 (a pre-trained model) with some added layers.  
Helps transfer knowledge from other image datasets.
### ⏱️ Step 8: Define Callbacks  
Adds features like:
- Early stopping: stop if model stops improving.
- ModelCheckpoint: save the best version of model.
- 
### ⚙️ Step 9: Compile Model  
Define loss function, optimizer (Adam), and evaluation metric (accuracy).  



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.keras import TqdmCallback
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers, optimizers, callbacks
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.applications import EfficientNetV2B2
import tensorflow as tf
import pickle

# Data Augmentation
data_augmentation = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

# Load Pretrained EfficientNetV2B2 with matching input size
base_model = EfficientNetV2B2(
    include_top=False,
    input_shape=(224, 224, 3),
    include_preprocessing=True,
    weights='imagenet'
)

# Freeze early layers
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

# Build Final Model
model = Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    base_model,
    GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(6, activation='softmax')
])

# Compile Model
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
early = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = callbacks.ModelCheckpoint(
    filepath='best_model224.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# Pre-training Summary
epochs = 15
print("\n📦 Starting Model Training...")
print(f"Epochs       : {epochs}")
print(f"Class Weights: {class_weights}")
print(f"Checkpoint   : Best model saved to 'best_model224.keras'")
print(f"Early Stop   : Enabled with patience=3\n")

# Train the Model with Tqdm progress bar
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    class_weight=class_weights,
    batch_size=32,
    callbacks=[early, checkpoint, TqdmCallback(verbose=1)]
)

# Display training log as a table
df_log = pd.DataFrame(history.history)
print("\n📊 Training Log Summary:")
print(df_log.to_string(index=True))

# Save history for later comparison
with open("history224.pkl", "wb") as f:
    pickle.dump(history.history, f)

# Plot Accuracy and Loss Graphs
plt.figure(figsize=(12, 4))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc', marker='o')
plt.plot(history.history['val_accuracy'], label='Val Acc', marker='o')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True)
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', marker='o')
plt.plot(history.history['val_loss'], label='Val Loss', marker='o')
plt.title('Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# 👁️ View Base Model Summary (EfficientNetV2B2 without classification head)
print("\n📌 Base Model (EfficientNetV2B2) Summary:")
base_model.summary()


In [ ]:
# 👁️ View Final Model Summary (with Data Augmentation, GAP, Dropout, Dense)
print("\n📌 Final Model (Full Architecture) Summary:")
model.summary()


### 🧠 Step 10: Train the Model  
Fit the model to training data.  
Validates on validation dataset.  


In [ ]:
# Evaluate the best saved model on the test set
model = tf.keras.models.load_model('best_model224.keras')

# Evaluate
loss, acc = model.evaluate(test_ds_eval, verbose=2)
print(f"\n✅ Test Accuracy : {acc * 100:.2f}%")
print(f"📉 Test Loss     : {loss:.4f}")


### 🧠 Step 10: Train the Model  
Fit the model to training data.  
Validates on validation dataset.  


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Get true labels and predictions
y_true = []
y_pred = []

for images, labels in test_ds_eval:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

# Classification Report
print("\n📄 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)


### 📈 Step 11: Training Graphs  
Plots training vs validation accuracy/loss.  
Helps visualize overfitting or underfitting.  
### 📏 Step 12: Evaluate on Test Data  
Evaluate model on unseen data.  
Returns accuracy, loss, etc.  
### 🔍 Step 13: Confusion Matrix  
Visualizes correct and incorrect predictions across classes.  


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
import random

# Take a batch from test set
for images, labels in test_ds.take(1):
    preds = model.predict(images)
    pred_labels = np.argmax(preds, axis=1)

    plt.figure(figsize=(12, 8))
    for i in range(6):
        idx = random.randint(0, len(images) - 1)
        plt.subplot(2, 3, i + 1)
        plt.imshow(images[idx].numpy().astype("uint8"))
        plt.title(f"True: {class_names[labels[idx]]}\nPred: {class_names[pred_labels[idx]]}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
import tensorflow as tf
import pickle
import pandas as pd

# === Paths ===
model_path = 'best_model224.keras'       # Change to your actual model file if needed
history_path = 'history224.pkl'          # Change to your actual history file if needed

# === Load the model ===
model = tf.keras.models.load_model(model_path)
print(f"✅ Loaded model from: {model_path}")

# === Load training history ===
with open(history_path, 'rb') as f:
    history = pickle.load(f)

# === Extract final epoch metrics ===
final_accuracy     = history['accuracy'][-1]
final_val_accuracy = history['val_accuracy'][-1]
final_loss         = history['loss'][-1]
final_val_loss     = history['val_loss'][-1]

# === Print final summary ===
print("\n📊 Final Training Summary:")
print(f"Train Accuracy     : {final_accuracy:.4f}")
print(f"Validation Accuracy: {final_val_accuracy:.4f}")
print(f"Train Loss         : {final_loss:.4f}")
print(f"Validation Loss    : {final_val_loss:.4f}")


In [ ]:
import pickle
import matplotlib.pyplot as plt

# Load history
with open("history224.pkl", "rb") as f:
    history = pickle.load(f)

# Accuracy values
acc = history['accuracy']
val_acc = history['val_accuracy']
epochs = range(len(acc))

# Plot Accuracy
plt.figure(figsize=(6, 4))
plt.plot(epochs, acc, label='Train Accuracy', color='green', marker='o')
plt.plot(epochs, val_acc, label='Validation Accuracy', color='blue', marker='s', linestyle='--')
plt.title('📈 Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.xticks(epochs)
plt.tight_layout()
plt.show()


In [ ]:
# Loss values
loss = history['loss']
val_loss = history['val_loss']
epochs = range(len(loss))

# Plot Loss
plt.figure(figsize=(6, 4))
plt.plot(epochs, loss, label='Train Loss', color='red', marker='o')
plt.plot(epochs, val_loss, label='Validation Loss', color='orange', marker='s', linestyle='--')
plt.title('📉 Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.xticks(epochs)
plt.tight_layout()
plt.show()


### 🌐 Step 14: Deploy with Gradio  
Creates a simple web interface where user can upload image and see prediction.  


In [ ]:
pip install --upgrade gradio

In [ ]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image

# 🚀 Load the trained model
model = tf.keras.models.load_model("best_model224.keras")

# 🏷️ Class labels
class_names = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

# 🔍 Prediction Function
def predict_image(image: Image.Image):
    image = image.resize((224, 224))
    img_array = tf.keras.utils.img_to_array(image) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array)[0]
    predicted_label = class_names[np.argmax(predictions)]
    confidence_scores = {class_names[i]: float(f"{predictions[i]:.4f}") for i in range(len(class_names))}

    return predicted_label, confidence_scores

# 🎨 Gradio Interface
interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="📸 Upload or Drag Image Here"),
    outputs=[
        gr.Label(label="Predicted Class"),
        gr.Label(label="Confidence Scores")
    ],
    title="🗑️ Garbage Classifier",
    description="Upload a garbage image to classify it into one of six categories: cardboard, glass, metal, paper, plastic, or trash.",
    theme="default"
)

# 🌐 Launch with public link
interface.launch(share=True)


